# 1. Import thư viện

In [1]:
import re
import os
import glob
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.dummy import DummyClassifier
import warnings

# Tắt các cảnh báo không cần thiết để đầu ra sạch sẽ hơn
warnings.filterwarnings('ignore')

In [3]:
from google.colab import drive
drive.mount('https://drive.google.com/drive/folders/1xMttqoTlBt_QM1kKHsjQIsMRMPP435kJ?usp=drive_link')

ValueError: Mountpoint must be in a directory that exists

# Softmax Regression.

In [ ]:
class RestaurantABSA:
    """
    Hệ thống phân tích ý kiến đa khía cạnh (ABSA) cho lĩnh vực Nhà hàng.
    Sử dụng Softmax Regression kết hợp TF-IDF đặc trưng n-gram.
    """

    def __init__(self):
        # Định nghĩa các thực thể và thuộc tính theo chuẩn VLSP 2018
        self.entities = ["RESTAURANT", "AMBIENCE", "LOCATION", "FOOD", "SERVICE", "DRINKS"]
        self.attributes = ["GENERAL", "PRICES", "QUALITY", "STYLE&OPTIONS", "MISCELLANEOUS"]
        self.categories = sorted([f"{e}#{a}" for e in self.entities for a in self.attributes])

        # Cấu hình Vectorizer để trích xuất đặc trưng văn bản
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)
        self.model_pipeline = {}
        self.optimal_params = {}

    def parse_raw_data(self, file_path):
        """Đọc tệp dữ liệu gốc và chuyển đổi sang định dạng danh sách."""
        corpus, annotations = [], []

        if not os.path.exists(file_path):
            print(f"Cảnh báo: Không tìm thấy tệp {file_path}")
            return corpus, annotations

        with open(file_path, 'r', encoding='utf-8') as stream:
            content = stream.read().strip().split('\n')

        # Cấu trúc tệp VLSP: ID, Văn bản, Nhãn, Dòng trống (tổng 4 dòng)
        for idx in range(0, len(content), 4):
            if idx + 2 >= len(content): break

            raw_text = content[idx+1].strip()
            label_str = content[idx+2].strip()

            # Trích xuất cặp Aspect-Polarity bằng Regular Expression
            found_tags = re.findall(r'\{([^,]+),\s*([^}]+)\}', label_str)
            sentiment_map = {asp.strip(): pol.strip() for asp in found_tags if asp.strip() in self.categories}

            corpus.append(raw_text)
            annotations.append(sentiment_map)

        return corpus, annotations

    def prepare_target_vectors(self, labels_list):
        """Tách nhãn thành các vector mục tiêu riêng cho từng mô hình Aspect."""
        y_grouped = {asp: [] for asp in self.categories}
        for entry in labels_list:
            for asp in self.categories:
                # Nếu khía cạnh không được nhắc tới, gán nhãn 'null'
                y_grouped[asp].append(entry.get(asp, 'null'))
        return y_grouped

    def train_and_optimize(self, train_x, train_y_dict, dev_x, dev_y_dict):
        """Huấn luyện và tìm tham số C tối ưu trên tập Validation."""
        print("--- Bắt đầu quy trình huấn luyện & tối ưu hóa ---")

        # Chuyển đổi văn bản sang ma trận TF-IDF
        x_train_tfidf = self.vectorizer.fit_transform(train_x)
        x_dev_tfidf = self.vectorizer.transform(dev_x)

        c_candidates = [0.01, 0.1, 1, 10, 100]

        for asp in self.categories:
            y_train = train_y_dict[asp]
            y_dev = dev_y_dict[asp]
            classes = np.unique(y_train)

            # Nếu dữ liệu chỉ có 1 nhãn (thường là 'null'), dùng DummyClassifier
            if len(classes) <= 1:
                clf = DummyClassifier(strategy='constant', constant=classes[0])
                clf.fit(x_train_tfidf, y_train)
                self.model_pipeline[asp] = clf
                self.optimal_params[asp] = "Default (Constant)"
                continue

            # Grid search tìm tham số C tốt nhất
            best_c, top_score = 1.0, -1.0
            for c in c_candidates:
                temp_clf = LogisticRegression(C=c, solver='lbfgs', multi_class='multinomial', max_iter=500)
                temp_clf.fit(x_train_tfidf, y_train)

                score = accuracy_score(y_dev, temp_clf.predict(x_dev_tfidf))
                if score > top_score:
                    top_score = score
                    best_c = c

            # Huấn luyện lại mô hình với tham số C tốt nhất cho Aspect đó
            final_clf = LogisticRegression(C=best_c, solver='lbfgs', multi_class='multinomial', max_iter=500)
            final_clf.fit(x_train_tfidf, y_train)

            self.model_pipeline[asp] = final_clf
            self.optimal_params[asp] = best_c

    def run_evaluation(self, test_x, test_y_dict):
        """Đánh giá hiệu năng trên tập kiểm thử."""
        x_test_tfidf = self.vectorizer.transform(test_x)
        results = []

        print(f"\n{'KHÍA CẠNH (ASPECT)':<30} | {'ACC':<10} | {'F1 MACRO':<10}")
        print("-" * 60)

        for asp in self.categories:
            y_true = test_y_dict[asp]
            y_pred = self.model_pipeline[asp].predict(x_test_tfidf)

            # Chỉ đánh giá nếu Aspect có xuất hiện trong thực tế (không phải toàn bộ là 'null')
            if len(set(y_true)) > 1 or 'null' not in set(y_true):
                acc = accuracy_score(y_true, y_pred)
                f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

                print(f"{asp:<30} | {acc:.4f}     | {f1:.4f}")
                results.append((acc, f1))

        if results:
            avg_acc = np.mean([r[0] for r in results])
            avg_f1 = np.mean([r[1] for r in results])
            print("-" * 60)
            print(f"{'CHỈ SỐ TRUNG BÌNH':<30} | {avg_acc:.4f}     | {avg_f1:.4f}")

# Chuẩn bị dữ liệu và Tiền xử lý

In [ ]:
# 1. Khởi tạo Engine
absa_engine = RestaurantABSA()

# 2. Đọc dữ liệu từ file (Đảm bảo tên file đúng với thư mục của bạn)
# Nếu dùng trên Drive, bạn cần mount drive và thay đổi path cho đúng
train_txt, train_lbl = absa_engine.parse_raw_data('train_data.txt')
dev_txt, dev_lbl = absa_engine.parse_raw_data('dev_data.txt')
test_txt, test_lbl = absa_engine.parse_raw_data('test_data.txt')

# 3. Phân rã nhãn cho từng Aspect
y_train_set = absa_engine.prepare_target_vectors(train_lbl)
y_dev_set = absa_engine.prepare_target_vectors(dev_lbl)
y_test_set = absa_engine.prepare_target_vectors(test_lbl)

print(f"Đã tải {len(train_txt)} mẫu Train, {len(dev_txt)} mẫu Dev, {len(test_txt)} mẫu Test.")

# Huấn luyện và Đánh giá

In [ ]:
# 4. Chạy quy trình huấn luyện và tối ưu hóa tham số
absa_engine.train_and_optimize(train_txt, y_train_set, dev_txt, y_dev_set)

# 5. Đánh giá mô hình trên tập dữ liệu Test độc lập
absa_engine.run_evaluation(test_txt, y_test_set)